# 2. Baseline для решения задачи классификации для датасета [Adult Census Income](https://archive.ics.uci.edu/dataset/2/adult)

In [1]:
# нужные импорты
import numpy as np
import pandas as pd

import seaborn as sns
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings('ignore')
sns.set(style='darkgrid')

In [2]:
# для начала подгрузим данные 
columns = ['age', 'workclass', 'fnlwgt', 'education', 'education-num',
           'marital-status', 'occupation', 'relationship', 'race', 'sex',
           'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'income']


train = pd.read_csv('data/adult.data', names=columns, skipinitialspace=True, na_values=['?'])
test = pd.read_csv('data/adult.test', names=columns, skipinitialspace=True, na_values=['?'], skiprows=1)
test['income'] = test['income'].str.rstrip('.') # тк в тесте точка на конце

In [3]:
# и заполним пропуски как мы решили это делать в EDA
train.fillna('Unknown', inplace=True)
test.fillna('Unknown', inplace=True)

In [4]:
# выделим целевую переменную
y_train = train['income'].apply(lambda x: x=='>50K').astype(int)
X_train = train.drop(columns='income', axis=1)

y_test = test['income'].apply(lambda x: x=='>50K').astype(int)
X_test = test.drop(columns='income', axis=1)

### Train-validation split

Для оценки baseline'ов исходные обучающие данные были поделены на train и val. Качество будет замеряться на валидации. Также при разбиении используем стратифицированное семплирование, чтобы сохранить исходный баланс классов в двух выборках.  

In [5]:
# отделим валидационную выборку
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, stratify=y_train, random_state=42)

Далее возьмем три модели:
- Logictic regression - линейный baseline 
- Random forest - сложная модель в основе которой решающее дерево
- кастомный GBM - также сложная модель на основе деревьев, но написанная в `boosting.py`
Замерим качество у каждой из них и сравним время обучения

### Preprocessing

Закодируем категориальные признаки через one-hot encoder и стандартизируем числовые признаки. Все preprocessing transformers для преобразования данных обучены на обучающих данных и потом применены к валиации и к тесту через transform(), чтобы избежать утечки данных

In [ ]:
# для начала нужно предобработать признаки: закодировать категориальные признаки и стандартизовать числовые
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler

# взяли с EDA
cat_features = train.select_dtypes(include=['object', 'category']).columns.tolist()
num_features = train.select_dtypes(include=['int64', 'float64']).columns.tolist()

# Исключаем целевую переменную
if 'income' in cat_features:
    cat_features.remove('income')

column_transformer = ColumnTransformer([
    ('ohe', OneHotEncoder(handle_unknown='ignore'), cat_features),
    ('scaling', StandardScaler(), num_features)
])

X_train_scaled = column_transformer.fit_transform(X_train)
X_val_scaled = column_transformer.transform(X_val)
X_test_scaled = column_transformer.transform(X_test)

### Baseline 1: Logistic Regression

Начнем с логистической регрессии как с линейного бэйзлайна. Эта модель не может в точности уловить нелинейные зависимости и взаимосвязи, но является хорошей отправной точкой для сравнения с tree-based методами.

Возьмем стандартную логистическую регрессию из sklearn и замерим ее качество. В качестве метрики возьмем ROC-AUC

In [12]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

logreg = LogisticRegression(random_state=42, max_iter=1000)

logreg.fit(X_train_scaled, y_train)
y_pred = logreg.predict_proba(X_val_scaled)[:, 1]
print(f'AUC-ROC (Logistic Regression): {roc_auc_score(y_val, y_pred):.4f}')

AUC-ROC (Logistic Regression): 0.9098


### Baseline 2: Random Forest

Далее оценим tree-based baseline — Random Forest. В отличие от логистической регрессии Random Forest может находить нелинейные зависимости между признаками, которые мы наблюдали ранее в EDA.

In [13]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(random_state=42)
rf.fit(X_train_scaled, y_train)
y_pred = rf.predict_proba(X_val_scaled)[:, 1]
print(f'AUC-ROC (Random forest): {roc_auc_score(y_val, y_pred):.4f}')

AUC-ROC (Random forest): 0.9067


### Baseline 3: Custom Gradient Boosting

И наконец, оценим кастомную реализацию бустинга из `boosting.py`. Эта модель использует логистическую функцию потерь для бинарной классификации, поэтому целевая переменная должна быть закодирована в виде {-1, 1}.

In [9]:
%load_ext autoreload

In [10]:
%autoreload 2

from boosting import Boosting

In [14]:
boosting = Boosting(
    verbose=False,
    random_state=42,
    n_estimators = 200, # по дефолту их 20, что очень мало, поэтому добавим
    base_model_params = {"max_depth": 3, "min_samples_leaf": 20} # похожие на дефолтные параметры из sklearn
)

# преобразуем таргет в 1/-1 
y_train_gbm = y_train.apply(lambda x: 1 if x == 1 else -1)
y_val_gbm = y_val.apply(lambda x: 1 if x == 1 else -1)

boosting.fit(X_train_scaled, y_train_gbm)
y_pred = boosting.predict_proba(X_val_scaled)[:, 1]
print(f'AUC-ROC (GBM): {roc_auc_score(y_val, y_pred):.4f}')

AUC-ROC (GBM): 0.9059


### Conclusion

Сравнение baseline'ов показало, что Логистическая регрессия, кастомный GBM и Random Forest достигают схожего ROC-AUC на валидации, около 0.9.

Такие результаты говорят о том, что данные содержат сильные линейные зависимости после обработки категориальных признаков через OHE, что делает выборку линейно разделимой. Также конкурентоспособность кастомного бустинга указывает на то, что данная реализация способна выявлять значимые закономерности и может быть дополнительно усовершенствована. Это служит отправной точкой для модификации кастомного бустинга и добавления к нему различных имплементаций, которые мы можем наблюдать в XGBoost, CatBoost или LightGBM.